
# W1
## Data Quality, Evaluation, Scaling, and Encoding

**Prothoma AKter**   

This is a small assignment that connects topics from Module 1, 2, and 3.  
You must complete it in this Colab notebook.

You will need to use concepts that appeared in the videos:
- Module 1 and 2: basic descriptive statistics, proportions, confusion matrix, accuracy, precision, recall
- Module 3: standardization, min max scaling, nominal vs ordinal, one hot encoding, ordinal encoding, Euclidean and Manhattan distance

Please do not use any extra libraries beyond `pandas`, `numpy`.



---
## 0. Setup and Dataset

We will use a dataset that should have columns given below:

- `user_id`  
- `age`  
- `monthly_income` (numeric)  
- `daily_screen_time_min` (numeric)  
- `daily_app_opens` (numeric)  
- `true_label` and `pred_label` for a binary classification task (0 or 1)  
- `satisfaction_level` (for example: `Low`, `Medium`, `High`)  
- `city_type` (for example: `Urban`, `Suburban`, `Rural`)


In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np

In [2]:
# Cell 2: Load the dataset (Already done for you)
df = pd.read_csv("https://drive.google.com/uc?export=download&id=1OmDDCh4MD1TtvAemnwVDyz5zwCIXJ220")

# Show first few rows
df.head()

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type
0,1,43,3734.19,109,48,0,0,Medium,Suburban
1,2,49,2594.19,194,7,0,0,Low,Urban
2,3,19,3550.47,146,36,1,0,High,Rural
3,4,19,3821.18,287,14,1,0,High,Suburban
4,5,63,1750.84,66,46,0,0,Medium,Suburban



### 0.1 Check your dataset

1. Confirm that the dataset loaded correctly.  
2. Check that you have at least these columns:  
   - numeric: `age`, `monthly_income`, `daily_screen_time_min`, `daily_app_opens`  
   - labels: `true_label`, `pred_label`  
   - categorical: `satisfaction_level`, `city_type`  



---
## Part A - Module 1 and 2 Review

In this part you will do simple descriptive statistics and basic classification evaluation.



### Q1. Descriptive statistics on a numeric feature

Choose one numeric column, for example `daily_screen_time_min`.


In [3]:
# Q1.1: Choose your numeric column here [We already write this ans]
num_col = "daily_screen_time_min"

df[num_col].describe()

,daily_screen_time_min
count,100.000000
mean,181.890000
std,68.886951
min,60.000000
25%,122.000000
50%,178.000000
75%,243.750000
max,299.000000



> **Q1.2 Short answer: [Marks: 05]**  
> Look at the count, mean, min, max, and standard deviation for your chosen column.  
> In 2 to 3 sentences, comment on what you see.  
> For example, does the max look very far from the mean, or does it look quite close?

Write your answer here:

>  The max value is way higher than the average, which really shows a heavy right-skew caused by some extreme outliers. Plus, that massive gap is pretty clear from the high standard deviation compared to the mean, meaning the data points are all over the place.
>  
>  



### Q2. Proportion of positive class

Use the `true_label` column, where 1 means "positive" and 0 means "negative".


In [4]:
# Q2.1: Compute proportion of positive class [We already write this ans]
label_col = "true_label"

positive_count = (df[label_col] == 1).sum()
total_count = df.shape[0]
positive_proportion = positive_count / total_count

print("Positive count:", positive_count)
print("Total samples:", total_count)
print("Proportion of positive class:", positive_proportion)

Positive count: 52
Total samples: 100
Proportion of positive class: 0.52



> **Q2.2 Short answer: [5 marks]**  
> In 1 to 2 sentences, explain what this proportion tells you about your dataset.  
> For example, is the dataset balanced between 0 and 1, or is one class much more common?

Write your answer here:

>  Since positive instances make up 52% of the total samples, the dataset is pretty well-balanced between the two classes. Because that split is super close to a 50-50 ratio, neither class really dominates or is way more common than the other.
>  
>  



### Q3. Confusion matrix and basic metrics

For this question, use:
- `true_label` as the actual label  
- `pred_label` as the model prediction


In [5]:
# Q3.1: Manually compute TP, TN, FP, FN [We already write this ans]
true_col = "true_label"
pred_col = "pred_label"

tp = ((df[true_col] == 1) & (df[pred_col] == 1)).sum()
tn = ((df[true_col] == 0) & (df[pred_col] == 0)).sum()
fp = ((df[true_col] == 0) & (df[pred_col] == 1)).sum()
fn = ((df[true_col] == 1) & (df[pred_col] == 0)).sum()

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

TP: 28
TN: 27
FP: 21
FN: 24


In [6]:
# Q3.2: Compute accuracy, precision, recall [We already write this ans]
accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.55
Precision: 0.5714285714285714
Recall: 0.5384615384615384



> **Q3.3 Short answer: [10 marks]**  
> In 3 to 4 sentences, briefly comment on the model using these three metrics.  
> For example, is the model catching most positives (high recall) or being careful when it predicts positive (high precision)?

Write your answer here:

>  > The model achieves an overall accuracy of 0.55, meaning it correctly predicts the label slightly more than half of the time. Its precision of approximately 0.57 indicates that when the model forecasts a positive outcome, it is accurate nearly 57% of the time. Meanwhile, the recall of approximately 0.54 shows that the model successfully identifies about 54% of all actual positive cases in the dataset.
>
>
>  
>  



---
## Part B - Module 3: Scaling and Encoding

Now we will pick a few features and apply scaling and encoding.



### Q4. Standardization and Min max scaling

Use one numeric column, `monthly_income`.


In [7]:
df.head(1)

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type
0,1,43,3734.19,109,48,0,0,Medium,Suburban


In [8]:
# Q4.1: Choose the numeric column [2 marks]
numerical_col =df["monthly_income"]
numerical_col.head()

,monthly_income
0,3734.19
1,2594.19
2,3550.47
3,3821.18
4,1750.84


In [9]:
# Q4.2: Standardization with z-score [10 marks]
mean=numerical_col.mean()
std=numerical_col.std()
z_score=(numerical_col-mean)/std
z_score=z_score.round(2)
z_score.head()

,monthly_income
0,0.94
1,-0.32
2,0.74
3,1.04
4,-1.26


In [10]:
# Q4.3: Min max scaling implementation [10 marks]
min_v=numerical_col.min()
max_v=numerical_col.max()
min_max_scaled=(numerical_col-min_v)/(max_v-min_v)
min_max_scaled=min_max_scaled.round(2)
min_max_scaled.head()

,monthly_income
0,0.68
1,0.39
2,0.63
3,0.70
4,0.19



> **Q4.4 Short answer: [3 marks]**  
> Compare the standardized and min max scaled columns in 2 to 3 sentences.  
> Mention what kind of range each one uses and how the numbers look.

Write your answer here:

>  Standardization transforms the data so it has a mean of 0 and a standard deviation of 1, resulting in a mix of positive and negative numbers. On the other hand, min-max scaling squishes all the values into a strict range between 0 and 1. Even though they look different, both methods keep the original shape of the data intact while shifting them to completely new scales.
>  
>  



### Q5. One hot and ordinal encoding

We will use:
- `city_type` as a nominal feature  
- `satisfaction_level` as an ordinal feature with order `Low` < `Medium` < `High`  


In [11]:
# Q5.1: One hot encoding for city_type using pandas [10 marks]
df_encoded=pd.get_dummies(df["city_type"],prefix='city_type',dtype=int)
df_encoded.head()

,city_type_Rural,city_type_Suburban,city_type_Urban
0,0,1,0
1,0,0,1
2,1,0,0
3,0,1,0
4,0,1,0


In [12]:
# Q5.2: Attach one hot encoded columns to df [5 marks]
df=df.drop('city_type',axis=1)
df=pd.concat([df,df_encoded],axis=1)
df.head()

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type_Rural,city_type_Suburban,city_type_Urban
0,1,43,3734.19,109,48,0,0,Medium,0,1,0
1,2,49,2594.19,194,7,0,0,Low,0,0,1
2,3,19,3550.47,146,36,1,0,High,1,0,0
3,4,19,3821.18,287,14,1,0,High,0,1,0
4,5,63,1750.84,66,46,0,0,Medium,0,1,0


In [13]:
# Q5.3: Ordinal encoding for satisfaction_level [10 marks]
ordinal_encoding={'Medium':2, 'Low':1, 'High':3}
df['satisfaction_level']=df['satisfaction_level'].map(ordinal_encoding).astype(int)
df.head()

,user_id,age,monthly_income,daily_screen_time_min,daily_app_opens,true_label,pred_label,satisfaction_level,city_type_Rural,city_type_Suburban,city_type_Urban
0,1,43,3734.19,109,48,0,0,2,0,1,0
1,2,49,2594.19,194,7,0,0,1,0,0,1
2,3,19,3550.47,146,36,1,0,3,1,0,0
3,4,19,3821.18,287,14,1,0,3,0,1,0
4,5,63,1750.84,66,46,0,0,2,0,1,0



> **Q5.4 Short answer: [5 marks]**  
> In 2 to 3 sentences, explain why one hot encoding is suitable for `city_type`  
> and why ordinal encoding is suitable for `satisfaction_level`.

Write your answer here:

>  One-hot encoding works best for city_type because it's a nominal feature where categories like Suburban or Urban have no inherent numerical order. On the other hand, ordinal encoding fits satisfaction_level perfectly since levels like low, medium, and high carry a natural ranking and progression.
>  
>  



---
## Part C - Module 3: Distances between users

For this small part we will work with vectors based on scaled numeric features.



### Q6. Euclidean and Manhattan distance

Build 2D vectors for user 0 and user 1 using:
- `income_std`  
- `daily_app_opens` (or its min max scaled version if you prefer)


In [14]:
# Q6.1: Build 2D vectors for first two users [We already write this ans]
vec_cols = ["monthly_income", "daily_app_opens"]

v1 = df.loc[0, vec_cols].values
v2 = df.loc[1, vec_cols].values

print("v1:", v1)
print("v2:", v2)

v1: [3734.19   48.  ]
v2: [2594.19    7.  ]


In [15]:
# Q6.2: Euclidean distance computation [5 marks]
euclidean_distance =np.linalg.norm(v1-v2)
print("Euclidean distance:", euclidean_distance)

Euclidean distance: 1140.7370424422975


In [16]:
# Q6.3: Manhattan distance computation [5 marks]
manhattan_distance=np.linalg.norm(v1-v2,ord=1)
print("Manhattan distance:", manhattan_distance)

Manhattan distance: 1181.0



> **Q6.4 Short answer: [5 marks]**  
> Which one is larger in your result, Euclidean or Manhattan distance  
> and why does that usually happen based on their formulas?

Write your answer here:

>  Manhattan distance is larger than Euclidean distance in this result because it sums the absolute axis-wise differences instead of taking the shortest straight-line path. Geometrically, traveling along grid lines (Manhattan) is always greater than or equal to the direct diagonal distance (Euclidean).
>  
>  



---
## Final Reflection [10 marks]

> In 4 to 6 sentences, describe how the three modules connect in this assignment.  
> Mention:
> - One idea from Module 1 or 2 that you used  
> - One idea from Module 3 that you used  
> - How these ideas together help you understand a dataset more deeply

Write your reflection here:

>  This assignment connects data preprocessing, feature encoding, and distance metrics into a unified machine learning workflow.I used one-hot and ordinal encoding techniques to transform categorical attributes like city types and satisfaction levels into numerical formats.I then applied vector distance metrics, such as Euclidean and Manhattan distances, to compute mathematical similarities between data points.Together, these techniques convert raw features into structured vectors, enabling a deeper and more precise analysis of the dataset.
>  
>  



## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।
